# 03 · Dos líneas base y cinco familias

Las dos bases se ejecutan primero para t+3. Se usa AP (PR-AUC no interpolada), misma preparación y CV para todas las familias. Defaults sklearn; solo se fija semilla. La regla estacional reproduce PSet 1 y su score no es una probabilidad calibrada.

In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image, Markdown
from src.datos import DATA, DOCS, EVIDENCE
from src.datos import execute
HAS_DATA = (DATA / "obt.parquet").exists()
display(Markdown("**OBT real local disponible**" if HAS_DATA else
    "**PENDIENTE:** faltan credenciales Snowflake válidas y snapshot. Este notebook no contiene resultados empíricos del ECU 911."))


## Comparación registrada

Familias: logística (lineal), random forest (bagging), hist gradient boosting (boosting), MLP (red neuronal), GaussianNB (probabilístico). Train AP, CV media±SD, brecha y tiempo permiten diagnosticar generalización. No se garantiza superar ambas bases.

In [ ]:
if HAS_DATA:
    rows = execute("candidatos")
    results = pd.DataFrame(rows)
    display(results[["horizon", "phase", "name", "train_ap", "cv_ap_mean", "cv_ap_std", "gap", "learning_rate", "fit_seconds", "warnings"]])
    display(results.loc[results.phase.eq("candidato"), ["horizon", "name", "beats_both_baselines"]])
    _ = execute("report")
    display(Image(filename=str(EVIDENCE / "comparacion_modelos.png")))


## Evidencia de aporte de features

Ablación de ratios con regresión logística y los mismos folds. Una diferencia inferior a la SD no prueba aporte; se conserva el resultado negativo si ocurre.

In [ ]:
if HAS_DATA:
    display(pd.DataFrame(execute("ablacion")))

## LLM mediante prompting

El caso sí puede plantearse como clasificación tabular zero-shot. Se prueba en una muestra del último fold, con salidas JSON y sin fechas/nombres exactos. Se compara con las predicciones OOF de las mismas filas. No se usa hold-out ni se incorpora una muestra pequeña al ranking de CV completa. Se usa `qwen2.5:1.5b` en Ollama local, sin API key; la evaluación requiere la OBT real.

In [ ]:
if HAS_DATA:
    display(execute("llm"))
else:
    display(Markdown("**Evaluación LLM pendiente:** Ollama local con qwen2.5:1.5b está configurado; falta el snapshot real. El prompt está en src/datos.py (llm_trial)."))